In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os
import time
import numpy as np
import pandas as pd
import multiprocessing
from joblib import Parallel, delayed
from ripser import ripser
from numpy import loadtxt

# Configuration
MAX_FILE_SIZE_MB = 1.5


def load_data(file_path):

    ext = file_path.split(".")[-1].lower()

    try:
        if ext == "txt":
            return loadtxt(file_path)

        elif ext == "csv":
            df = pd.read_csv(file_path)
            if "Unnamed: 0" in df.columns:
                df = df.drop(columns="Unnamed: 0")
            return df.values

        else:
            raise ValueError(f"Unsupported file type: {ext}")

    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None


def run_rips_on_file(tube, patient, input_dir, output_dir):
    """
    Run Vietoris-Rips on a file.
    """
    if tube.startswith(".") or "Others" in tube:
        return

    input_path = os.path.join(input_dir, patient, tube)
    output_base = os.path.join(output_dir, patient, tube)

    # ---
    try:
        size_mb = os.path.getsize(input_path) / 1e6
    except Exception as e:
        print(f"Error reading size for {input_path}: {e}")
        return

    # ---
    if os.path.exists(output_base + "_Rips0.txt"):
        print(f"Skipped (already done): {patient}/{tube}")
        return

    # ---
    if size_mb > MAX_FILE_SIZE_MB:
        print(f"Skipped (too large): {patient}/{tube} ({size_mb:.2f} MB)")
        return

    # ---
    data = load_data(input_path)
    if data is None:
        return

    # ---
    print(f"Running Rips on {patient}/{tube}")
    start = time.time()

    try:
        diagrams = ripser(data)["dgms"]
    except Exception as e:
        print(f"Error running ripser on {input_path}: {e}")
        return

    elapsed = time.time() - start
    print(f"Finished in {elapsed:.2f}s")

    # ---
    try:
        np.savetxt(output_base + "_Rips0.txt", diagrams[0], fmt="%f")
        np.savetxt(output_base + "_Rips1.txt", diagrams[1], fmt="%f")
    except Exception as e:
        print(f"Error saving results for {input_path}: {e}")


def process_patient(patient, input_dir, output_dir):
    """
    Process all of a patient's files.
    """
    if patient.startswith("."):
        return

    print(f"\n--- Processing patient: {patient} ---")

    patient_path = os.path.join(input_dir, patient)
    output_path = os.path.join(output_dir, patient)

    os.makedirs(output_path, exist_ok=True)

    try:
        tubes = sorted(os.listdir(patient_path))
    except Exception as e:
        print(f"Error listing {patient_path}: {e}")
        return

    for tube in tubes:
        run_rips_on_file(tube, patient, input_dir, output_dir)


def run_parallel(input_dir, output_dir):
    """
    Run the analysis in parallel for each patient.
    """
    os.makedirs(output_dir, exist_ok=True)

    try:
        patients = sorted(os.listdir(input_dir))
    except Exception as e:
        print(f"Error reading input directory: {e}")
        return

    num_cores = multiprocessing.cpu_count()

    Parallel(n_jobs=num_cores)(
        delayed(process_patient)(patient, input_dir, output_dir)
        for patient in patients
    )

# MAIN

if __name__ == "__main__":
    base_input = "outputfolder"
    base_output = "outputfolder/RIPS"

    groups = ["Relapse", "NonRelapse"]

    for group in groups:
        input_dir = os.path.join(base_input, group)
        output_dir = os.path.join(base_output, group)

        print(f"\nProcessing {group}")
        run_parallel(input_dir, output_dir)